# Reconstruccion del Dataset Completo 2019-2025 desde Fuentes Crudas

**Problema resuelto:** Las versiones anteriores mezclaban valores crudos (2019-2020) con z-scores (2021-2025), causando doble normalizacion y MAEs artificialmente bajos.

**Solucion:** Reconstruir TODO desde las fuentes crudas originales con la misma agregacion (suma de distritos a nivel provincia-mes) para ambos periodos.

| Paso | Fuente | Archivos |
|------|--------|----------|
| 1 | MIDAGRI | `Sisagri_2016_2025.xlsx` hojas 2016_2020 + 2021_2025 |
| 2 | NASA POWER | `clima_dataset_2019_2020.csv` + `clima_dataset_final.csv` |
| 3 | INDECI | `indeci_temporal_2019_2025.csv` + `indeci_temporal_2021_2025.csv` |
| 4 | NLP | `sentimiento_mensual_v2.csv` (solo 2021+) |
| 5 | Merge + Normalizacion unica | StandardScaler fit en train |

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print(f'ROOT: {ROOT}')

ROOT: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-


---
## PASO 1 — MIDAGRI completo 2019-2025 (valores crudos)

In [2]:
MIDAGRI_PATH = ROOT / 'sources/midagri/Sisagri_2016_2025.xlsx'

def load_midagri_limon(sheet, year_filter):
    df = pd.read_excel(MIDAGRI_PATH, sheet_name=sheet)
    mask = (
        df['dsc_Cultivo'].str.contains('LIMON', case=False, na=False)
        & ~df['dsc_Cultivo'].str.contains('LIMA|PAJA', case=False, na=False)
        & df['anho'].isin(year_filter)
    )
    filtered = df[mask].copy()
    agg = (filtered.groupby(['anho', 'mes', 'Dpto', 'Prov'])
           .agg({'PRODUCCION(t)': 'sum', 'MTO_PRECCHAC (S/ x kg)': 'mean'})
           .reset_index()
           .rename(columns={
               'Dpto': 'departamento', 'Prov': 'provincia',
               'PRODUCCION(t)': 'produccion_t',
               'MTO_PRECCHAC (S/ x kg)': 'precio_chacra_kg'
           }))
    return agg

midagri_1920 = load_midagri_limon('2016_2020', [2019, 2020])
midagri_2125 = load_midagri_limon('2021_2025', [2021, 2022, 2023, 2024, 2025])
midagri = pd.concat([midagri_1920, midagri_2125], ignore_index=True)

print(f'MIDAGRI 2019-20: {midagri_1920.shape}')
print(f'MIDAGRI 2021-25: {midagri_2125.shape}')
print(f'MIDAGRI total:   {midagri.shape}')

# Verificar consistencia de escala entre periodos
for period, m in [('2019-20', midagri_1920), ('2021-25', midagri_2125)]:
    print(f'\n  {period}:')
    print(f'    produccion_t:    mean={m.produccion_t.mean():.1f}  median={m.produccion_t.median():.1f}  std={m.produccion_t.std():.1f}')
    print(f'    precio_chacra:   mean={m.precio_chacra_kg.mean():.3f}  std={m.precio_chacra_kg.std():.3f}')
    print(f'    dptos: {m.departamento.nunique()}  provs: {m.provincia.nunique()}')

MIDAGRI 2019-20: (1913, 6)
MIDAGRI 2021-25: (4530, 6)
MIDAGRI total:   (6443, 6)

  2019-20:
    produccion_t:    mean=316.0  median=18.6  std=1249.8
    precio_chacra:   mean=1.206  std=0.781
    dptos: 23  provs: 104

  2021-25:
    produccion_t:    mean=342.5  median=20.0  std=1363.1
    precio_chacra:   mean=1.713  std=1.150
    dptos: 23  provs: 105


---
## PASO 2 — NASA POWER completo 2019-2025

In [3]:
nasa_1920 = pd.read_csv(ROOT / 'data/interim/nasa/clima_dataset_2019_2020.csv')
nasa_2125 = pd.read_csv(ROOT / 'data/interim/nasa/clima_dataset_final.csv')

COLS_CLIMA = ['ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M',
              'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon']

for df_n in [nasa_1920, nasa_2125]:
    df_n['DATE'] = pd.to_datetime(df_n['DATE'])
    df_n['anho'] = df_n['DATE'].dt.year
    df_n['mes']  = df_n['DATE'].dt.month

nasa = pd.concat([
    nasa_1920[['departamento', 'provincia', 'anho', 'mes'] + COLS_CLIMA],
    nasa_2125[['departamento', 'provincia', 'anho', 'mes'] + COLS_CLIMA],
], ignore_index=True)

print(f'NASA 2019-20: {nasa_1920.shape}')
print(f'NASA 2021-25: {nasa_2125.shape}')
print(f'NASA total:   {nasa.shape}')
print(f'Columnas iguales: {nasa_1920.columns.tolist() == nasa_2125.columns.tolist()}')

# Verificar unidades consistentes
for period, mask in [('2019-20', nasa['anho'].isin([2019,2020])),
                     ('2021-25', nasa['anho'] >= 2021)]:
    subset = nasa[mask]
    print(f'\n  {period}:')
    for c in ['T2M', 'PRECTOTCORR', 'ALLSKY_SFC_SW_DWN']:
        print(f'    {c:25s}  mean={subset[c].mean():.2f}  std={subset[c].std():.2f}')

print(f'\nProvs NASA: {nasa.groupby(["departamento","provincia"]).ngroups}')

NASA 2019-20: (2448, 17)
NASA 2021-25: (6120, 17)
NASA total:   (8568, 14)
Columnas iguales: True

  2019-20:
    T2M                        mean=18.00  std=6.44
    PRECTOTCORR                mean=1.92  std=2.59
    ALLSKY_SFC_SW_DWN          mean=18.09  std=3.07

  2021-25:
    T2M                        mean=18.35  std=6.57
    PRECTOTCORR                mean=1.83  std=2.49
    ALLSKY_SFC_SW_DWN          mean=17.89  std=3.10

Provs NASA: 102


---
## PASO 3 — INDECI completo 2019-2025

In [4]:
COLS_INDECI = ['num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas']

# Cargar ambos archivos INDECI
indeci_1920 = pd.read_csv(ROOT / 'data/interim/indeci/indeci_temporal_2019_2025.csv')
indeci_2123 = pd.read_csv(ROOT / 'data/interim/indeci/indeci_temporal_2021_2025.csv')

# Extraer anho y mes
for df_i in [indeci_1920, indeci_2123]:
    df_i['anho'] = df_i['fecha_evento'].str[:4].astype(int)
    df_i['mes']  = df_i['fecha_evento'].str[5:7].astype(int)

# Limpiar encoding en ambos
ENCODING_MAP = {
    'APURA\x8dMAC': 'APURIMAC', 'HUA\x81NUCO': 'HUANUCO',
    'JUNA\x8dN': 'JUNIN', 'SAN MARTA\x8dN': 'SAN MARTIN',
}
PROV_MAP = {
    'ALTOAMAZONAS': 'ALTO AMAZONAS', 'MARISCALRAMONCASTILLA': 'MARISCAL RAMON CASTILLA',
    'GENERALSANCHEZCERRO': 'GENERAL SANCHEZ CERRO', 'MARISCALCACERES': 'MARISCAL CACERES',
    'MARISCALNIETO': 'MARISCAL NIETO', 'SANCHEZCARRION': 'SANCHEZ CARRION',
    'PAUCARDELSARASARA': 'PAUCAR DEL SARA SARA', 'GRANCHIMU': 'GRAN CHIMU',
    'SANIGNACIO': 'SAN IGNACIO', 'SANMARCOS': 'SAN MARCOS',
    'SANMIGUEL': 'SAN MIGUEL', 'SANTACRUZ': 'SANTA CRUZ',
    'SANMARTIN': 'SAN MARTIN', 'LACONVENCION': 'LA CONVENCION',
    'LAUNION': 'LA UNION', 'LAMAR': 'LA MAR', 'ELDORADO': 'EL DORADO',
    'PADREABAD': 'PADRE ABAD', 'PUERTOINCA': 'PUERTO INCA',
    'CORONELPORTILLO': 'CORONEL PORTILLO', 'VILCASHUAMAN': 'VILCAS HUAMAN',
    'CONTRALMIRA NTEVILLAR': 'CONTRALMIRANTE VILLAR',
    'LEONCIOP RADO': 'LEONCIO PRADO',
}

for df_i in [indeci_1920, indeci_2123]:
    for col in ['departamento', 'provincia']:
        for bad, good in ENCODING_MAP.items():
            df_i[col] = df_i[col].str.replace(bad, good, regex=False)
        df_i[col] = df_i[col].str.strip().str.upper()
    for bad, good in PROV_MAP.items():
        df_i['provincia'] = df_i['provincia'].str.replace(bad, good, regex=False)

indeci = pd.concat([
    indeci_1920[['departamento', 'provincia', 'anho', 'mes'] + COLS_INDECI],
    indeci_2123[['departamento', 'provincia', 'anho', 'mes'] + COLS_INDECI],
], ignore_index=True)

print(f'INDECI 2019-20:  {len(indeci_1920)} filas (rango: {indeci_1920.fecha_evento.min()} -> {indeci_1920.fecha_evento.max()})')
print(f'INDECI 2021-23:  {len(indeci_2123)} filas (rango: {indeci_2123.fecha_evento.min()} -> {indeci_2123.fecha_evento.max()})')
print(f'INDECI total:    {len(indeci)} filas')
print(f'Cobertura: 2019-01 a 2023-02. Meses 2023-03 a 2025-08 se rellenan con 0 en el merge (left join).')

INDECI 2019-20:  2749 filas (rango: 2019-01 -> 2020-12)


INDECI 2021-23:  3064 filas (rango: 2021-01 -> 2023-02)
INDECI total:    5813 filas
Cobertura: 2019-01 a 2023-02. Meses 2023-03 a 2025-08 se rellenan con 0 en el merge (left join).


---
## PASO 4 — NLP sentimiento

In [5]:
NLP_PATH = ROOT / 'notebooks/fase2/output/01_nlp_sentimiento/sentimiento_mensual_v2.csv'
df_nlp = pd.read_csv(NLP_PATH, encoding='utf-8-sig')
fc = [c for c in df_nlp.columns if any(k in c.lower() for k in ['fecha', 'periodo'])][0]
df_nlp = df_nlp.rename(columns={fc: 'fecha_evento'})
df_nlp['fecha_evento'] = pd.to_datetime(df_nlp['fecha_evento'])
df_nlp['nlp_index']      = df_nlp['avg_sentiment'] * np.log1p(df_nlp['n_noticias_beto'])
df_nlp['nlp_index_lag1'] = df_nlp['nlp_index'].shift(1).fillna(0)

print(f'NLP: {df_nlp.shape}  rango: {df_nlp.fecha_evento.min().date()} -> {df_nlp.fecha_evento.max().date()}')
print(f'Columnas: {df_nlp.columns.tolist()}')
print(f'nlp_index: mean={df_nlp.nlp_index.mean():.4f}  std={df_nlp.nlp_index.std():.4f}')

NLP: (60, 8)  rango: 2021-01-01 -> 2025-12-01
Columnas: ['fecha_evento', 'avg_sentiment', 'n_noticias_beto', 'n_positivas', 'n_negativas', 'n_neutrales', 'nlp_index', 'nlp_index_lag1']
nlp_index: mean=-0.1429  std=0.4692


---
## PASO 5 — Merge, agregacion mensual y normalizacion unica

In [6]:
# 5a: Merge MIDAGRI + NASA (inner join)
MERGE_KEYS = ['departamento', 'provincia', 'anho', 'mes']
merged = midagri.merge(nasa, on=MERGE_KEYS, how='inner')
print(f'MIDAGRI + NASA: {merged.shape}')

# 5b: + INDECI (left join, rellenar 0 donde no hay emergencias)
merged = merged.merge(indeci, on=MERGE_KEYS, how='left')
for col in COLS_INDECI:
    merged[col] = merged[col].fillna(0)
print(f'+ INDECI: {merged.shape}')

# 5c: Construir fecha_evento y agregar a serie mensual
merged['fecha_evento'] = pd.to_datetime(
    merged['anho'].astype(str) + '-' + merged['mes'].astype(str).str.zfill(2) + '-01')

df_monthly = (merged.drop(columns=['anho', 'mes'])
              .groupby('fecha_evento').mean(numeric_only=True)
              .reset_index()
              .sort_values('fecha_evento')
              .reset_index(drop=True))

print(f'\nSerie mensual: {df_monthly.shape}')
print(f'Rango: {df_monthly.fecha_evento.min().date()} -> {df_monthly.fecha_evento.max().date()}')
print(f'Meses: {len(df_monthly)}')

# 5d: Incorporar NLP
df_monthly = df_monthly.merge(
    df_nlp[['fecha_evento', 'nlp_index', 'nlp_index_lag1']],
    on='fecha_evento', how='left')
df_monthly['nlp_index']      = df_monthly['nlp_index'].fillna(0)
df_monthly['nlp_index_lag1'] = df_monthly['nlp_index_lag1'].fillna(0)
df_monthly['tiene_nlp'] = (df_monthly['fecha_evento'].dt.year >= 2021).astype(int)

# es_shock sobre valores crudos reales
TARGET = 'produccion_t'
df_monthly['es_shock'] = (df_monthly[TARGET].pct_change().abs() > 0.20).astype(int)
df_monthly.loc[0, 'es_shock'] = 0

# Columnas temporales ciclicas
df_monthly['mes_num']       = df_monthly['fecha_evento'].dt.month
df_monthly['month_sin']     = np.sin(2 * np.pi * df_monthly['mes_num'] / 12)
df_monthly['month_cos']     = np.cos(2 * np.pi * df_monthly['mes_num'] / 12)
df_monthly['trimestre_num'] = (df_monthly['mes_num'] - 1) // 3 + 1
df_monthly['trimestre_sin'] = np.sin(2 * np.pi * df_monthly['trimestre_num'] / 4)
df_monthly['trimestre_cos'] = np.cos(2 * np.pi * df_monthly['trimestre_num'] / 4)

print(f'\nDataset mensual final: {df_monthly.shape}')
print(f'Columnas: {df_monthly.columns.tolist()}')

# Verificar consistencia de produccion entre periodos
print(f'\n--- produccion_t CRUDA por periodo (media provincial mensual) ---')
for label, y_list in [('2019-2020', [2019,2020]), ('2021-2022', [2021,2022]),
                       ('2023-2024', [2023,2024]), ('2025', [2025])]:
    mask = df_monthly['fecha_evento'].dt.year.isin(y_list)
    if mask.sum() == 0: continue
    vals = df_monthly.loc[mask, TARGET]
    print(f'  {label:10s}  n={mask.sum():>3}  mean={vals.mean():8.1f}  std={vals.std():8.1f}  min={vals.min():8.1f}  max={vals.max():8.1f}')

MIDAGRI + NASA: (6106, 16)
+ INDECI: (6106, 19)

Serie mensual: (80, 16)
Rango: 2019-01-01 -> 2025-08-01
Meses: 80

Dataset mensual final: (80, 26)
Columnas: ['fecha_evento', 'produccion_t', 'precio_chacra_kg', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'nlp_index', 'nlp_index_lag1', 'tiene_nlp', 'es_shock', 'mes_num', 'month_sin', 'month_cos', 'trimestre_num', 'trimestre_sin', 'trimestre_cos']

--- produccion_t CRUDA por periodo (media provincial mensual) ---
  2019-2020   n= 24  mean=   332.2  std=    83.4  min=   173.3  max=   482.6
  2021-2022   n= 24  mean=   385.9  std=    94.3  min=   238.1  max=   581.8
  2023-2024   n= 24  mean=   388.0  std=   122.6  min=   145.1  max=   604.5
  2025        n=  8  mean=   233.2  std=    50.1  min=   169.7  max=   301.7


In [7]:
# 5e: Split y normalizacion UNICA
n_total = len(df_monthly)
n_test  = 12
n_train = n_total - n_test

df_train = df_monthly.iloc[:n_train].copy()
df_test  = df_monthly.iloc[n_train:].copy()

NO_SCALE = ['fecha_evento', 'tiene_nlp', 'es_shock', TARGET]
COLS_SCALE = [c for c in df_monthly.columns if c not in NO_SCALE]

scaler_feat = StandardScaler()
scaler_feat.fit(df_train[COLS_SCALE])

scaler_tgt = StandardScaler()
scaler_tgt.fit(df_train[[TARGET]])

df_train_sc = df_train.copy()
df_test_sc  = df_test.copy()
df_train_sc[COLS_SCALE] = scaler_feat.transform(df_train[COLS_SCALE])
df_test_sc[COLS_SCALE]  = scaler_feat.transform(df_test[COLS_SCALE])
df_train_sc[TARGET] = scaler_tgt.transform(df_train[[TARGET]])
df_test_sc[TARGET]  = scaler_tgt.transform(df_test[[TARGET]])

dataset_final = pd.concat([df_train_sc, df_test_sc], ignore_index=True)

print(f'Split: train={n_train} ({df_train.fecha_evento.min().date()} -> {df_train.fecha_evento.max().date()})')
print(f'       test ={n_test} ({df_test.fecha_evento.min().date()} -> {df_test.fecha_evento.max().date()})')
print(f'\nTrain target: mean={df_train_sc[TARGET].mean():.6f}  std={df_train_sc[TARGET].std():.4f}')
print(f'Test  target: mean={df_test_sc[TARGET].mean():.4f}  std={df_test_sc[TARGET].std():.4f}')

# Verificar distribucion del target por periodo DENTRO del train
mask_tr19 = df_train_sc['fecha_evento'].dt.year.isin([2019, 2020])
mask_tr21 = df_train_sc['fecha_evento'].dt.year >= 2021

print(f'\n--- TARGET escalado por periodo en train ---')
print(f'  2019-20 ({mask_tr19.sum()}m): mean={df_train_sc.loc[mask_tr19, TARGET].mean():+.4f}  std={df_train_sc.loc[mask_tr19, TARGET].std():.4f}')
print(f'  2021-24 ({mask_tr21.sum()}m): mean={df_train_sc.loc[mask_tr21, TARGET].mean():+.4f}  std={df_train_sc.loc[mask_tr21, TARGET].std():.4f}')

# 5f: Guardar
output_path = ROOT / 'data/processed/master_reconstruido_completo.csv'
dataset_final.to_csv(output_path, index=False)

scaler_params = {
    'features': COLS_SCALE,
    'mean': {c: float(m) for c, m in zip(COLS_SCALE, scaler_feat.mean_)},
    'scale': {c: float(s) for c, s in zip(COLS_SCALE, scaler_feat.scale_)},
    'target': TARGET,
    'target_mean': float(scaler_tgt.mean_[0]),
    'target_scale': float(scaler_tgt.scale_[0]),
    'n_train': n_train, 'n_test': n_test,
    'train_range': f'{df_train.fecha_evento.min().date()} -> {df_train.fecha_evento.max().date()}',
    'test_range': f'{df_test.fecha_evento.min().date()} -> {df_test.fecha_evento.max().date()}',
    'fuente': 'Reconstruido desde MIDAGRI+NASA+INDECI crudos, normalizacion unica',
}

scaler_path = ROOT / 'resultados/scaler_reconstruido.json'
with open(scaler_path, 'w') as f:
    json.dump(scaler_params, f, indent=2)

print(f'\nGuardado: {output_path.name} ({output_path.stat().st_size/1024:.1f} KB)')
print(f'Scaler:  {scaler_path.name}')

Split: train=68 (2019-01-01 -> 2024-08-01)
       test =12 (2024-09-01 -> 2025-08-01)

Train target: mean=-0.000000  std=1.0074
Test  target: mean=-0.7008  std=0.9341

--- TARGET escalado por periodo en train ---
  2019-20 (24m): mean=-0.3243  std=0.7953
  2021-24 (44m): mean=+0.1769  std=1.0735

Guardado: master_reconstruido_completo.csv (36.9 KB)
Scaler:  scaler_reconstruido.json


In [8]:
# 5g: Reporte final
print('='*75)
print('REPORTE FINAL — DATASET RECONSTRUIDO')
print('='*75)
print(f'Archivo:   {output_path.name}')
print(f'Filas:     {len(dataset_final)} ({n_train} train + {n_test} test)')
print(f'Columnas:  {len(dataset_final.columns)}')
print(f'Rango:     {dataset_final.fecha_evento.min()} -> {dataset_final.fecha_evento.max()}')
print(f'Nulos:     {dataset_final.isnull().sum().sum()}')

print(f'\n--- Scaler target ---')
print(f'  mean = {scaler_tgt.mean_[0]:.2f} t  (produccion media provincial mensual)')
print(f'  scale = {scaler_tgt.scale_[0]:.2f} t')

print(f'\n--- Produccion real (desnormalizada) por periodo ---')
for label, y_list in [('2019-2020', [2019,2020]), ('2021-2022', [2021,2022]),
                       ('2023-2024', [2023,2024]), ('2025 (test)', [2025])]:
    mask = dataset_final['fecha_evento'].dt.year.isin(y_list)
    if mask.sum() == 0: continue
    z = dataset_final.loc[mask, TARGET]
    real = z * scaler_tgt.scale_[0] + scaler_tgt.mean_[0]
    print(f'  {label:15s}  n={mask.sum():>3}  z: mean={z.mean():+.3f} std={z.std():.3f}  |  real: mean={real.mean():.1f}t  std={real.std():.1f}t')

print(f'\n--- Shocks en test ---')
test_slice = dataset_final.iloc[-n_test:]
n_shocks = test_slice['es_shock'].sum()
print(f'  {n_shocks} de {n_test} meses')
shock_months = test_slice[test_slice['es_shock']==1]['fecha_evento'].dt.strftime('%Y-%m').tolist()
print(f'  Meses: {shock_months}')

print(f'\n--- Primeras 3 filas ---')
display(dataset_final.head(3))
print(f'\n--- Ultimas 3 filas ---')
display(dataset_final.tail(3))
print('='*75)

REPORTE FINAL — DATASET RECONSTRUIDO
Archivo:   master_reconstruido_completo.csv
Filas:     80 (68 train + 12 test)
Columnas:  26
Rango:     2019-01-01 00:00:00 -> 2025-08-01 00:00:00
Nulos:     0

--- Scaler target ---
  mean = 366.17 t  (produccion media provincial mensual)
  scale = 104.86 t

--- Produccion real (desnormalizada) por periodo ---
  2019-2020        n= 24  z: mean=-0.324 std=0.795  |  real: mean=332.2t  std=83.4t
  2021-2022        n= 24  z: mean=+0.188 std=0.899  |  real: mean=385.9t  std=94.3t
  2023-2024        n= 24  z: mean=+0.208 std=1.169  |  real: mean=388.0t  std=122.6t
  2025 (test)      n=  8  z: mean=-1.268 std=0.478  |  real: mean=233.2t  std=50.1t

--- Shocks en test ---
  1 de 12 meses
  Meses: ['2025-01']

--- Primeras 3 filas ---


,fecha_evento,produccion_t,precio_chacra_kg,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,T2M,T2M_MAX,T2M_MIN,...,nlp_index,nlp_index_lag1,tiene_nlp,es_shock,mes_num,month_sin,month_cos,trimestre_num,trimestre_sin,trimestre_cos
0,2019-01-01,0.349241,-0.918982,-0.585649,1.330968,1.079470,1.239586,0.240983,0.011853,0.734777,...,0.304853,0.275792,0,0,-1.549276,0.658698,1.275497,-1.294024,1.373655,0.063457
1,2019-02-01,0.638451,-0.933457,-1.550283,1.333624,1.573021,1.675052,0.233561,-0.671732,1.043129,...,0.304853,0.275792,0,0,-1.255000,1.176964,0.757231,-1.294024,1.373655,0.063457
2,2019-03-01,0.529930,-0.885296,-0.886094,1.240009,1.056151,1.588358,-0.453397,-1.207996,0.428771,...,0.304853,0.275792,0,0,-0.960724,1.366662,0.049267,-1.294024,1.373655,0.063457



--- Ultimas 3 filas ---


,fecha_evento,produccion_t,precio_chacra_kg,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,T2M,T2M_MAX,T2M_MIN,...,nlp_index,nlp_index_lag1,tiene_nlp,es_shock,mes_num,month_sin,month_cos,trimestre_num,trimestre_sin,trimestre_cos
77,2025-06-01,-1.873942,-0.014198,-2.565675,1.542335,-0.173080,1.137436,-1.426908,-1.710991,-0.975930,...,0.988349,0.673188,1,0,-0.077897,-0.049267,-1.366662,-0.386873,-0.020502,-1.374911
78,2025-07-01,-1.589560,-0.019733,-0.268587,-1.331105,-1.205530,-0.233295,-1.325887,-1.043518,-1.777159,...,1.100537,1.001313,1,0,0.216379,-0.757231,-1.176964,0.520278,-1.414659,0.063457
79,2025-08-01,-1.594468,1.164898,0.226802,-1.096185,-0.775746,-0.492259,-0.211745,-0.066186,-0.243290,...,0.660261,1.120398,1,0,0.510655,-1.275497,-0.658698,0.520278,-1.414659,0.063457
